# Change in the 22/12 optical state across elevation pairs, from Danish 1.3 CWFS (v2)

**Author:** Aaron Roodman  
**Date Created:** 2026-10-29  
**Last Modified:** 2026-10-29  
**Status:** In Progress  
**Keywords:** elevation, look-up table, LUT, bounce test, v-mode 1, Danish 1.3, Trim, optical state

## Description

Uses the elevation pairs of v1: consecutive same-night science visits at most
`MAX_DT_MIN` apart that change elevation by at least 10 deg, with azimuth and camera
rotator angle each within 5 deg and the band unchanged. For each visit:

1. Reads the commanded Trim for every degree of freedom (DOF) from the `value_added`
   database.
2. Recovers the deviation DOF in the 22/12 scheme (22 DOF, 12 v-modes) from the Danish 1.3
   corner wavefront sensor (CWFS) deviation (total minus intrinsic, OCS), from the
   `cwfs_science` product. Only visits with Danish 1.3 Zernikes on all four corners are kept.
3. Forms the optical state, Trim minus deviation, in DOF and v-mode space.

Then tabulates the change across each pair (later minus earlier visit) in the 22 DOF and
12 v-modes, and fits v-mode 1 against the change in elevation.

The optical state excludes the hexapod look-up table (LUT), so its change across a pair is
the elevation dependence the LUT does not already correct. That is the quantity the
elevation bounce test measures.

**Output:** `pairs_state.parquet`, `pairs_state.csv`, `slopes.csv` and
`elevation_pairs_state.pdf` in the dated run directory.

**Based on:** `elevation_pairs_v1.ipynb` (pair search), `aos/bounce`

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-10-29 | Aaron Roodman | Initial version |
| 2026-10-29 | Aaron Roodman | Default to dt <= 5 min, non-consecutive pairs |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Optical State per Visit](#analysis)
6. [Results & Plots](#results)

<a id='params'></a>
## Parameters

In [1]:
from rubinwork.common import study_run

DAY_OBS_RANGE = None          # (lo, hi) as YYYYMMDD, or None for every night
MIN_DELTA_EL_DEG = 10.0       # minimum |elevation change| [deg]
MAX_DELTA_AZ_DEG = 5.0        # maximum |azimuth change| [deg], wrapped to +-180
MAX_DELTA_ROT_DEG = 5.0       # maximum |camera rotator angle change| [deg]
MAX_DT_MIN = 5.0              # maximum time between the two visit starts [min]
CONSECUTIVE_ONLY = False      # keep only pairs of consecutive science visits
CWFS_VARIANT = 'd13-blitz_rso1071'   # cwfs_science variant: Danish 1.3 blitz unpaired
DOF_SET, N_MODES = 'standard_22', 12  # the 22/12 scheme

run = study_run('aos/elevation_pairs', f'state_d13_22_12_dt{MAX_DT_MIN:g}min' + ('' if CONSECUTIVE_ONLY else '_all'))

<a id='setup'></a>
## Setup & Imports

In [2]:
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from scipy import stats
import statsmodels.api as sm

from rubinwork import aos_state
from rubinwork.common.utils import nmad, repo_root, setup_plotting
from rubinwork.products import cwfs_science
from rubinwork.products import value_added as efd_db

sys.path.insert(0, str(repo_root() / 'aos' / 'elevation_pairs' / 'code'))
import pairs_lib as PL

warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
setup_plotting()

<a id='functions'></a>
## Helper Functions

Pair search and state recovery are in `code/pairs_lib.py`.

In [3]:
DOF_IDX = aos_state.DOF_SETS[DOF_SET]
DOF_LABEL = {0: 'M2 dz', 1: 'M2 dx', 2: 'M2 dy', 3: 'M2 rx', 4: 'M2 ry',
             5: 'Cam dz', 6: 'Cam dx', 7: 'Cam dy', 8: 'Cam rx', 9: 'Cam ry',
             **{k: f'M1M3 B{k - 9}' for k in range(10, 30)},
             **{k: f'M2 B{k - 29}' for k in range(30, 50)}}
#: Hexapod tilts are stored in deg; every other DOF in um.
DOF_UNIT = {k: ('deg' if k in (3, 4, 8, 9) else 'um') for k in range(50)}


def huber_fit(x, y):
    """Huber robust line ``y = a + b x``.

    Parameters
    ----------
    x, y : `numpy.ndarray`

    Returns
    -------
    fit : `dict`
        ``slope``, ``slope_err``, ``intercept``, ``intercept_err`` (units of y per unit x,
        and of y), Pearson ``r`` and Spearman ``rho`` (dimensionless), ``n``, and
        ``nmad_resid`` (units of y).
    """
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    res = sm.RLM(y, sm.add_constant(x), M=sm.robust.norms.HuberT()).fit()
    return dict(slope=res.params[1], slope_err=res.bse[1], intercept=res.params[0],
                intercept_err=res.bse[0], r=stats.pearsonr(x, y)[0],
                rho=stats.spearmanr(x, y)[0], n=len(x),
                nmad_resid=nmad(y - res.fittedvalues))

<a id='data'></a>
## Data Access

Science visits with pointing and band from ConsDB, the Trim (`dof0`..`dof49`) and hexapod LUT from `value_added`, and the Danish 1.3 per-CCD Zernikes from `cwfs_science`.

In [4]:
trim_cols = [f'dof{k}' for k in range(50)]
lut_cols = [f'lut_dof{k}' for k in range(10)]
vis = efd_db.visits(day_obs_range=DAY_OBS_RANGE,
                    columns=['visit_id', 'day_obs', 'seq_num'] + trim_cols + lut_cols)
vis = efd_db.join_consdb(vis, groups=('meta',))
sci = vis[vis['img_type'] == 'science'].dropna(
    subset=['band', 'obs_start_mjd', 'altitude_deg', 'azimuth_deg_consdb', 'rotator_angle_deg'])
pairs = PL.find_pairs(sci, MIN_DELTA_EL_DEG, MAX_DELTA_AZ_DEG, MAX_DELTA_ROT_DEG,
                      MAX_DT_MIN, CONSECUTIVE_ONLY)
print(f'{len(sci)} science visits; {len(pairs)} pairs over {pairs["day_obs"].nunique()} nights')

ccd = cwfs_science.load_table('ccd_visits', CWFS_VARIANT)
pair_visits = np.unique(np.r_[pairs['visit_id_1'], pairs['visit_id_2']])
z_dev = PL.corner_deviation(ccd, pair_visits)
print(f'{len(z_dev)} of {len(pair_visits)} pair visits have a complete {CWFS_VARIANT} '
      f'corner deviation; {CWFS_VARIANT} spans day_obs {ccd["day_obs"].min()} to '
      f'{ccd["day_obs"].max()}')

        Use pytest instead. [astropy.tests.runner]
        Use pytest instead.


        Use pytest instead. [astropy.utils.decorators]
        Use pytest instead.


95938 science visits; 5812 pairs over 175 nights


5423 of 7292 pair visits have a complete d13-blitz_rso1071 corner deviation; d13-blitz_rso1071 spans day_obs 20251021 to 20260714


<a id='analysis'></a>
## Optical State per Visit

The 22/12 recovery uses `aos_state.recover_optical_state` (truncated SVD at 12 modes). The state is Trim minus deviation (`open_loop.optical_state_dofs`).

In [5]:
se = aos_state.make_state_estimator(dof_set=DOF_SET, n_modes=N_MODES)
per_visit = PL.recover_states(z_dev, sci.set_index('visit_id'), se, N_MODES, DOF_SET)
print(f'{len(per_visit)} visits with a recovered state and a Trim; median subspace residual '
      f'{per_visit["resid_rms_um"].median():.4f} um of wavefront')

names = ([f'{p}_dof{k}' for p in ('state', 'trim', 'dev') for k in DOF_IDX]
         + [f'{p}_v{m}' for p in ('state', 'trim', 'dev') for m in range(1, N_MODES + 1)])
pairs = PL.attach_deltas(pairs, per_visit, names + ['resid_rms_um'])
pairs = PL.attach_deltas(pairs, sci.set_index('visit_id'), lut_cols)
sel = pairs['d_state_v1'].notna()
ps = pairs[sel].reset_index(drop=True)
n_vis = len(np.unique(np.r_[ps['visit_id_1'], ps['visit_id_2']]))
print(f'{len(ps)} of {len(pairs)} pairs have a state at both visits, over '
      f'{ps["day_obs"].nunique()} nights and {n_vis} distinct visits')
# Non-consecutive pairs share visits, so the pairs are not independent and the formal fit
# errors below are too small; the distinct-visit count bounds the independent information.

5423 visits with a recovered state and a Trim; median subspace residual 0.0943 um of wavefront
4291 of 5812 pairs have a state at both visits, over 130 nights and 5394 distinct visits


<a id='results'></a>
## Results & Plots

Change across each pair, later minus earlier visit. v-modes are dimensionless; DOF in um, hexapod tilts in deg.

In [6]:
pd.set_option('display.width', 250)
show = ['day_obs', 'seq_num_1', 'seq_num_2', 'band', 'el_1_deg', 'd_el_deg', 'd_rot_deg',
        'dt_min', 'd_state_v1', 'd_trim_v1', 'd_dev_v1']
print('Angles [deg], dt [min], v-mode 1 [dimensionless]')
print(ps[show].round(4).to_string(max_rows=40))

Angles [deg], dt [min], v-mode 1 [dimensionless]
       day_obs  seq_num_1  seq_num_2 band  el_1_deg  d_el_deg  d_rot_deg  dt_min  d_state_v1  d_trim_v1  d_dev_v1
0     20251026        257        263    i   48.3338  -10.0644    -0.6000  3.9176     -0.0263    -0.0090    0.0173
1     20251026        257        264    i   48.3338  -10.0675    -0.4265  4.5471     -0.0225    -0.0129    0.0095
2     20251026        258        263    i   48.3262  -10.0568    -0.8042  3.2895     -0.0154    -0.0055    0.0099
3     20251026        258        264    i   48.3262  -10.0598    -0.6307  3.9190     -0.0116    -0.0094    0.0022
4     20251026        264        271    i   38.2664   11.0641     3.5307  4.4561      0.0561     0.0158   -0.0403
5     20251026        266        273    i   41.0472   11.0245     3.9802  4.4548      0.0115     0.0023   -0.0092
6     20251027        394        400    r   55.1606  -11.1936    -1.0411  4.0639     -0.0218    -0.0058    0.0161
7     20251027        394        401   

In [7]:
tab = ps[['day_obs', 'seq_num_1', 'seq_num_2', 'd_el_deg']
         + [f'd_state_dof{k}' for k in DOF_IDX]].copy()
tab.columns = ['day_obs', 'seq_1', 'seq_2', 'dEl'] + [f'{k}:{DOF_LABEL[k]}' for k in DOF_IDX]
print('Change in optical-state DOF across each pair (um; hexapod tilts deg), dEl in deg')
print(tab.round(3).to_string(max_rows=40))

Change in optical-state DOF across each pair (um; hexapod tilts deg), dEl in deg
       day_obs  seq_1  seq_2     dEl  0:M2 dz  1:M2 dx  2:M2 dy  3:M2 rx  4:M2 ry  5:Cam dz  6:Cam dx  7:Cam dy  8:Cam rx  9:Cam ry  10:M1M3 B1  11:M1M3 B2  12:M1M3 B3  13:M1M3 B4  14:M1M3 B5  15:M1M3 B6  16:M1M3 B7  30:M2 B1  31:M2 B2  32:M2 B3  33:M2 B4  34:M2 B5
0     20251026    257    263 -10.064  108.547    0.594  -11.289    0.000   -0.000   -98.078   415.796  -694.185    -0.002     0.000      -0.006      -0.000      -0.074       0.008      -0.006      -0.003       0.048     0.005     0.022    -0.011    -0.005    -0.054
1     20251026    257    264 -10.067   -1.996   -1.080   -2.979    0.000   -0.000     4.313   819.812  -935.258    -0.002    -0.001      -0.028       0.018      -0.104       0.015       0.005      -0.010       0.054    -0.030     0.077     0.010    -0.015    -0.072
2     20251026    258    263 -10.057   78.043   -8.986   27.069   -0.001   -0.001   -73.572   -31.220  -470.121    -0.002

In [8]:
rows = []
for kind in ('state', 'trim', 'dev'):
    for m in range(1, N_MODES + 1):
        f = huber_fit(ps['d_el_deg'].to_numpy(), ps[f'd_{kind}_v{m}'].to_numpy())
        rows.append(dict(term=f'{kind}_v{m}', unit='1/deg', **f))
    for k in DOF_IDX:
        f = huber_fit(ps['d_el_deg'].to_numpy(), ps[f'd_{kind}_dof{k}'].to_numpy())
        rows.append(dict(term=f'{kind}_dof{k} {DOF_LABEL[k]}', unit=f'{DOF_UNIT[k]}/deg', **f))
slopes = pd.DataFrame(rows)
slopes['signif'] = slopes['slope'] / slopes['slope_err']
print('Huber fit of the change against the elevation change, d = a + b dEl.')
print('slope b per deg of elevation; intercept a in the term\'s units; r, rho, signif = b/err '
      'all dimensionless')
cols = ['term', 'unit', 'slope', 'slope_err', 'signif', 'intercept', 'r', 'rho', 'n', 'nmad_resid']
print(slopes[slopes['term'].str.startswith('state')][cols].round(5).to_string(index=False))

Huber fit of the change against the elevation change, d = a + b dEl.
slope b per deg of elevation; intercept a in the term's units; r, rho, signif = b/err all dimensionless
               term    unit     slope  slope_err    signif  intercept        r      rho    n  nmad_resid
           state_v1   1/deg  -0.00025    0.00003  -8.83947   -0.00088 -0.13196 -0.12617 4291     0.02127
           state_v2   1/deg   0.00012    0.00003   3.76343    0.00206  0.03385  0.03467 4291     0.02501
           state_v3   1/deg  -0.00068    0.00004 -17.30531   -0.00401 -0.17458 -0.23139 4291     0.02963
           state_v4   1/deg   0.00029    0.00003   8.57818    0.00219  0.08540  0.10405 4291     0.02665
           state_v5   1/deg   0.00135    0.00004  31.36615    0.00460  0.36536  0.37685 4291     0.03319
           state_v6   1/deg  -0.00025    0.00014  -1.84258   -0.00054 -0.05492 -0.02325 4291     0.09952
           state_v7   1/deg  -0.00026    0.00014  -1.89143   -0.00472 -0.03227 -0.03026 4291

In [9]:
v1 = {k: huber_fit(ps['d_el_deg'].to_numpy(), ps[f'd_{k}_v1'].to_numpy())
      for k in ('state', 'trim', 'dev')}
for k, f in v1.items():
    print(f'{k:>5} v1: slope {f["slope"]:+.3e} +- {f["slope_err"]:.1e} per deg of elevation, '
          f'intercept {f["intercept"]:+.4f} (dimensionless), Pearson r {f["r"]:+.3f}, '
          f'Spearman rho {f["rho"]:+.3f}, n = {f["n"]} pairs, nMAD residual '
          f'{f["nmad_resid"]:.4f} (dimensionless)')

state v1: slope -2.474e-04 +- 2.8e-05 per deg of elevation, intercept -0.0009 (dimensionless), Pearson r -0.132, Spearman rho -0.126, n = 4291 pairs, nMAD residual 0.0213 (dimensionless)
 trim v1: slope +1.746e-05 +- 9.9e-06 per deg of elevation, intercept -0.0003 (dimensionless), Pearson r -0.011, Spearman rho +0.013, n = 4291 pairs, nMAD residual 0.0075 (dimensionless)
  dev v1: slope +2.664e-04 +- 2.2e-05 per deg of elevation, intercept +0.0005 (dimensionless), Pearson r +0.178, Spearman rho +0.168, n = 4291 pairs, nMAD residual 0.0170 (dimensionless)


In [10]:
pairs.to_parquet(run.path / 'pairs_state.parquet', index=False)
csv_cols = (show + ['visit_id_1', 'visit_id_2', 'el_2_deg', 'd_az_deg', 'rot_1_deg']
            + [f'd_state_dof{k}' for k in DOF_IDX] + [f'd_state_v{m}' for m in range(1, N_MODES + 1)])
ps[csv_cols].to_csv(run.path / 'pairs_state.csv', index=False, float_format='%.6g')
slopes.to_csv(run.path / 'slopes.csv', index=False, float_format='%.6g')


def scatter_fit(ax, x, y, f, ylabel, title):
    ax.scatter(x, y, s=4, alpha=0.5)
    xx = np.array([np.nanmin(x), np.nanmax(x)])
    ax.plot(xx, f['intercept'] + f['slope'] * xx, color='C3')
    ax.axhline(0, color='0.7', lw=0.5)
    ax.set_xlabel('elevation change, later minus earlier [deg]')
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=9)


with PdfPages(run.path / 'elevation_pairs_state.pdf') as pdf:
    fig = plt.figure(figsize=(8.5, 11))
    lines = [f'Elevation pairs: 22/12 optical state from {CWFS_VARIANT}',
             '',
             f'Cuts: |dEl| >= {MIN_DELTA_EL_DEG} deg, |dAz| < {MAX_DELTA_AZ_DEG} deg, '
             f'|dRot| < {MAX_DELTA_ROT_DEG} deg, same band,',
             f'      dt <= {MAX_DT_MIN} min, consecutive science visits only = {CONSECUTIVE_ONLY}',
             f'Pairs: {len(pairs)} found, {len(ps)} with a Danish 1.3 state at both visits, '
             f'{ps["day_obs"].nunique()} nights, {n_vis} distinct visits',
             'Pairs share visits when non-consecutive: formal errors are optimistic',
             'State = Trim - deviation (no hexapod LUT); change = later - earlier visit',
             '',
             'v-mode 1 change against elevation change, Huber fit d = a + b dEl:']
    for k, f in v1.items():
        lines.append(f'  {k:>5}: b = {f["slope"]:+.3e} +- {f["slope_err"]:.1e} /deg, '
                     f'r = {f["r"]:+.3f}, rho = {f["rho"]:+.3f}, n = {f["n"]}')
    lines += ['', 'Optical-state slopes per deg of elevation (signif = b / err):']
    st = slopes[slopes['term'].str.startswith('state')]
    for _, r in st.iterrows():
        lines.append(f'  {r["term"]:<22} {r["slope"]:+11.4g} +- {r["slope_err"]:9.2g} '
                     f'{r["unit"]:<8} signif {r["signif"]:+6.1f}  r {r["r"]:+.2f}')
    fig.text(0.05, 0.97, '\n'.join(lines), fontsize=7.5, va='top', family='monospace')
    pdf.savefig(fig)
    plt.close(fig)

    fig, axs = plt.subplots(1, 3, figsize=(14, 4.5), sharey=False)
    for ax, (k, lab) in zip(axs, [('state', 'optical state (Trim - deviation)'),
                                  ('trim', 'Trim'), ('dev', 'deviation')]):
        scatter_fit(ax, ps['d_el_deg'], ps[f'd_{k}_v1'], v1[k],
                    'v-mode 1 change [dimensionless]',
                    f'{lab}: slope {v1[k]["slope"]:+.2e}/deg, r = {v1[k]["r"]:+.2f}, '
                    f'n = {v1[k]["n"]}')
    fig.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 5))
    sc = ax.scatter(ps['el_1_deg'], ps['el_1_deg'] + ps['d_el_deg'], c=ps['d_state_v1'],
                    cmap='coolwarm', s=25)
    fig.colorbar(sc, label='optical-state v-mode 1 change [dimensionless]')
    ax.set_xlabel('elevation, earlier visit [deg]')
    ax.set_ylabel('elevation, later visit [deg]')
    pdf.savefig(fig)
    plt.close(fig)

    for kind, title in (('dof', 'optical-state DOF'), ('v', 'optical-state v-mode')):
        terms = ([(f'd_state_dof{k}', f'dof{k} {DOF_LABEL[k]}', DOF_UNIT[k]) for k in DOF_IDX]
                 if kind == 'dof' else
                 [(f'd_state_v{m}', f'v{m}', 'dimensionless') for m in range(1, N_MODES + 1)])
        ncol = 6 if kind == 'dof' else 4
        nrow = int(np.ceil(len(terms) / ncol))
        fig, axs = plt.subplots(nrow, ncol, figsize=(3.0 * ncol, 2.6 * nrow))
        for a, (col, lab, unit) in zip(axs.flat, terms):
            f = huber_fit(ps['d_el_deg'].to_numpy(), ps[col].to_numpy())
            scatter_fit(a, ps['d_el_deg'], ps[col], f, f'change [{unit}]',
                        f'{lab}: {f["slope"]:+.3g}/deg ({f["slope"] / f["slope_err"]:+.1f} sigma)')
            a.xaxis.label.set_size(7)
            a.yaxis.label.set_size(7)
        for a in list(axs.flat)[len(terms):]:
            a.axis('off')
        fig.suptitle(f'Change in {title} against elevation change, {len(ps)} pairs')
        fig.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)
print(f'Wrote {run.path}')

/lscratch/roodman/ipykernel_1680653/753077017.py:50: UserWarning: The figure layout has changed to tight
  fig.tight_layout()


/lscratch/roodman/ipykernel_1680653/753077017.py:79: UserWarning: The figure layout has changed to tight
  fig.tight_layout()


Wrote /sdf/group/rubin/u/roodman/LSST/rubin-work/studies/aos/elevation_pairs/20261029_state_d13_22_12_dt5min_all
